# Match_view: reanudar YOLO11s desde el checkpoint
Selecciona GPU T4 y ejecuta las dos celdas en orden. No ejecutes otra copia del entrenamiento al mismo tiempo.
Este cuaderno recupera optimizador, epoca y configuracion; verifica las imagenes y etiquetas originales.
El checkpoint original se conserva en una carpeta de recuperacion. No despliega pesos ni modifica la interfaz.
Estado de preparacion: codigo preparado y comprobado sintacticamente; reanudacion aun NO ejecutada.


In [ ]:
%pip install -q ultralytics==8.4.215 pyyaml
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'Selecciona GPU T4 antes de continuar'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import datetime, hashlib, importlib.util, json, shutil, subprocess, sys, traceback, zipfile
from ultralytics import YOLO
OUT = Path('/content/drive/MyDrive/Match_view_retraining_20261117T180409Z')
CHECKPOINT = OUT/'yolo11s_pretrained/weights/resumable.pt'
assert CHECKPOINT.is_file(), 'No se encuentra el checkpoint recuperable'
# Do not start a second concurrent training of this experiment.
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RECOVERY = OUT/('recovery_'+STAMP)
RECOVERY.mkdir()
shutil.copy2(CHECKPOINT, RECOVERY/'source_checkpoint.pt')
checkpoint_hash = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
checkpoint = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
completed = int(checkpoint.get('epoch', -1))+1
assert 0 < completed < 60, f'Checkpoint no reanudable: {completed} epocas'
assert checkpoint.get('optimizer') is not None, 'Falta el estado del optimizador'
print(f'Reanudar desde {completed}/60 epocas; no comenzar de cero', flush=True)
print('Checkpoint SHA256:', checkpoint_hash, flush=True)
del checkpoint
# Recreate the exact original dataset paths expected by checkpoint train_args.
BASE = Path('/content/match_view')
BASE.mkdir(exist_ok=True)
DATA = BASE/'audited_dataset'
REPO = Path('/content/repo_tmp')
SOURCE_REVISION = '30061bd41b2385d0cb94f6c325ffb2cc0042dd40'
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Josen-IG/Match_view.git',str(REPO)],check=True)
current = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
if current != SOURCE_REVISION:
    subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin',SOURCE_REVISION],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',SOURCE_REVISION],check=True)
# This is the exact experiment code saved with the original run.
spec = importlib.util.spec_from_file_location('experiment_recovery',OUT/'retrain_experiment.py')
experiment = importlib.util.module_from_spec(spec)
spec.loader.exec_module(experiment)
original_audit = json.loads((OUT/'audit.json').read_text())
if not DATA.exists():
    zips = [Path('/content/drive/MyDrive/match_view_training_images_20261117.zip'),
            Path('/content/drive/MyDrive/Git Hub/dataset_yolo_propio_images.zip')]
    ZIP = next((p for p in zips if p.is_file()), None)
    assert ZIP is not None, 'No se encuentra el ZIP en Mi unidad'
    with zipfile.ZipFile(ZIP) as z:
        assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
        z.extractall(BASE/('recovery_unpacked_'+STAMP))
    roots = list((BASE/('recovery_unpacked_'+STAMP)).rglob('images'))
    assert len(roots) == 1, roots
    DATA, audit = experiment.prepare(REPO,roots[0],BASE,RECOVERY)
    def identities(audit):
        return {(r['split'],r['stem']):(r['sha256_pixels'],r['sha256_source_labels']) for r in audit['manifest']}
    assert identities(audit) == identities(original_audit), 'Los datos difieren del experimento original'
# Verify existing data too, rather than trusting an interrupted runtime's directory.
for row in original_audit['manifest']:
    matches = list((DATA/'images'/row['split']).glob(row['stem']+'.*'))
    assert len(matches) == 1, row['stem']
    assert hashlib.sha256(matches[0].read_bytes()).hexdigest() == row['sha256_file'], row['stem']
    original = (REPO/'dataset/labels'/row['split']/(row['stem']+'.txt')).read_text()
    expected = [experiment.clipped_label(line)[0] for line in original.splitlines() if line.strip()]
    actual = (DATA/'labels'/row['split']/(row['stem']+'.txt')).read_text().splitlines()
    assert actual == expected, 'Etiquetas alteradas: '+row['stem']
train_entries = []
for row in original_audit['manifest']:
    if row['split'] == 'train':
        p = next((DATA/'images/train').glob(row['stem']+'.*'))
        train_entries += [str(p.resolve())]*(2 if row['has_ball'] else 1)
assert (DATA/'train_oversampled.txt').read_text().splitlines() == train_entries
print('Datos y etiquetas verificados contra los hashes originales',flush=True)
NAME = 'yolo11s_pretrained'
model = YOLO(str(CHECKPOINT))
def sync(trainer):
    shutil.copytree(trainer.save_dir,OUT/NAME,dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('epoch*.pt'))
    if Path(trainer.last).is_file():
        shutil.copy2(trainer.last,OUT/NAME/'weights/resumable.pt')
model.add_callback('on_model_save',sync)
experiment.write_json(RECOVERY/'provenance.json',{'checkpoint_sha256':checkpoint_hash,
    'completed_epochs':completed,'original_commit':SOURCE_REVISION,'production_changed':False})
try:
    experiment.write_json(OUT/'status.json',{'phase':'training','candidate':NAME,
        'resumed_from_epoch':completed,'completed':False})
    # resume=True restores optimizer, LR schedule, epoch counter and train arguments.
    model.train(resume=True)
    shutil.copytree(model.trainer.save_dir,OUT/NAME,dirs_exist_ok=True)
    best = OUT/NAME/'weights/best.pt'
    experiment.validate(best,DATA,OUT,NAME)
    experiment.fixed_threshold_counts(best,DATA,OUT,NAME)
    experiment.cpu_benchmark(best,DATA,OUT,NAME)
    assert all((OUT/('yolo11n_finetune_'+suffix+'.json')).is_file()
               for suffix in ('metrics','fixed_threshold','cpu'))
    experiment.write_json(OUT/'status.json',{'phase':'experiments_complete','completed':True,
        'deployed':False,'requires':'independent human-labelled holdout before deployment'})
except Exception:
    (RECOVERY/'failure.txt').write_text(traceback.format_exc())
    experiment.write_json(OUT/'status.json',{'phase':'recovery_failed','completed':False,
        'candidate':NAME,'recovery_dir':str(RECOVERY)})
    raise
print('Entrenamiento y evaluacion terminados. Modelo de produccion intacto.')
